# Using Optuna for hyperparameter tuning

# Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, LabelEncoder, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
import optuna
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    StackingClassifier
)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import lightgbm as lgb

In [2]:
o = pd.read_csv('/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv')
df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

In [3]:
df = pd.concat([df,o])
df.drop(columns = 'id',inplace = True)
df.sample(5)

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
13851,Silt,5.52,37.82,1.44,3.04,22.18,67.07,2238.71,9.72,5.79,Sugarcane,Vegetative,Rabi,Drip,Reservoir,2.71,Yes,109.64,South,Low
534290,Loamy,7.09,27.96,0.33,0.76,20.16,31.96,1153.64,6.03,11.93,Wheat,Flowering,Rabi,Rainfed,Rainwater,5.60,No,113.20,Central,Medium
277080,Clay,5.39,55.49,0.99,1.51,13.98,64.44,1962.60,4.72,16.64,Potato,Sowing,Kharif,Sprinkler,River,10.76,No,108.62,East,Low
174404,Sandy,6.81,25.25,0.66,0.83,13.74,46.78,2140.75,7.88,13.96,Cotton,Flowering,Rabi,Rainfed,Groundwater,4.46,Yes,0.37,South,Low
275940,Clay,5.43,13.03,0.45,2.98,38.34,94.48,1821.29,6.64,10.82,Wheat,Flowering,Rabi,Canal,Reservoir,7.59,Yes,46.76,West,Medium


# Feature Engineering

In [4]:
def add_cols(df):
    df['Moisture_Deficit'] = (40 - df['Soil_Moisture']).clip(lower=0)
    df['Weather_Stress'] = (
    df['Temperature_C'] * 0.3 +
    df['Wind_Speed_kmh'] * 0.2 -
    df['Humidity'] * 0.2 -
    df['Rainfall_mm'] * 0.3
    )

    stage_map = {
    "Sowing": 0,
    "Vegetative": 1,
    "Flowering": 2,
    "Harvest": 3
    }
    df['Stage_Weight'] = df['Crop_Growth_Stage'].map(stage_map)
    df['Rain_SoilMoisture'] = df['Rainfall_mm'] * df['Soil_Moisture']
    df['Mulch_Effect'] = (df['Mulching_Used'] == 'Yes').astype(int) * (1 - df['Soil_Moisture']/100)
    stage_mult = {'Sowing': 0.8, 'Vegetative': 1.2, 'Flowering': 1.8, 'Harvest': 0.6}
    df['Stage_Water_Multiplier'] = df['Crop_Growth_Stage'].map(stage_mult)
    return df

In [5]:
df = add_cols(df)
test = add_cols(test)
# df = df.fillna(0)

In [6]:
pd.set_option('display.max_columns', None)
df.sample(5)

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need,Moisture_Deficit,Weather_Stress,Stage_Weight,Rain_SoilMoisture,Mulch_Effect,Stage_Water_Multiplier
573317,Silt,6.68,58.20,1.46,1.11,30.78,30.92,1094.66,7.68,7.10,Potato,Harvest,Kharif,Drip,Groundwater,9.24,No,40.79,Central,Low,0.00,-323.928,3,63709.2120,0.0000,0.6
427168,Silt,7.95,30.52,0.46,0.70,15.64,28.76,1647.76,4.31,19.45,Rice,Sowing,Rabi,Canal,River,7.31,Yes,47.16,North,Low,9.48,-491.498,0,50289.6352,0.6948,0.8
478459,Clay,6.63,47.05,0.67,3.20,34.49,47.46,900.91,7.63,4.08,Cotton,Flowering,Kharif,Drip,Reservoir,0.31,No,24.80,East,Medium,0.00,-268.602,2,42387.8155,0.0000,1.8
598980,Silt,7.64,27.67,0.59,0.11,12.68,81.84,2110.02,8.19,6.52,Rice,Vegetative,Kharif,Canal,River,2.25,Yes,21.19,North,Low,12.33,-644.266,1,58384.2534,0.7233,1.2
282784,Loamy,7.24,28.40,0.77,1.71,18.62,75.35,1239.23,10.84,12.35,Sugarcane,Flowering,Rabi,Rainfed,Reservoir,11.96,No,16.61,West,Medium,11.60,-378.783,2,35194.1320,0.0000,1.8


# ColumnTransformer for OHE and Scaling

In [7]:
scaler_cols = df.select_dtypes(include = [int,float]).drop(columns = 'Stage_Weight').columns.tolist()
ohe_cols = df.select_dtypes(include = 'object').drop(columns = 'Irrigation_Need').columns.tolist()

In [8]:
clt = ColumnTransformer(transformers=[
    ('scaler',StandardScaler(),scaler_cols),
    ('ohe',OneHotEncoder(),ohe_cols)
],remainder='passthrough')
# clt = ColumnTransformer(transformers=[
#     ('scaler',MinMaxScaler(),scaler_cols),
#     ('ohe',OneHotEncoder(),ohe_cols)
# ],remainder='passthrough')

# Stacking

In [9]:
# data

X = df.drop(columns = 'Irrigation_Need')
X = clt.fit_transform(X)
y = df.Irrigation_Need
target_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
y = y.map(target_mapping)
y = np.array(y)
test_ = test.drop(columns = 'id')
test_ = clt.transform(test_)
n_classes = len(np.unique(y))
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features, {n_classes} classes")

Dataset: 640000 samples, 49 features, 3 classes


In [10]:
XGB = XGBClassifier(n_estimators = 831, max_depth = 4, learning_rate = 0.23590514823860093, min_child_weight = 3, gamma = 1.2456297224253232,
                         subsample = 0.7804018290363833, colsample_bytree = 0.7983030484833766, colsample_bylevel = 0.9241973849866497, reg_alpha = 0.2164667458085279,
                         reg_lambda = 6.479363279401208e-06,
                         objective = 'multi:softprob',
                         num_class = 3,
                         eval_metric = 'mlogloss',
                         random_state = 42,
                         device = 'cuda',
                         verbosity = 0,
                         class_weight = 'balanced')

In [11]:
# Level 1: 5 diverse high-performing base models
level1_estimators = [
    ('rf', RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_split=2,
        random_state=42, n_jobs=-1
    )),
    ('et', ExtraTreesClassifier(
        n_estimators=300, max_depth=None,
        random_state=43, n_jobs=-1
    )),
    ('cat', CatBoostClassifier(
        random_state=44,loss_function='MultiClass',
        eval_metric='MultiClass',
        auto_class_weights='Balanced'
    )),
    ('xgb1',XGB)
]

In [12]:
# Level 2: 3 strong meta-models (also diverse)
level2_estimators = [
    ('xgb2', XGB),
    ('lgb', lgb.LGBMClassifier(
        objective='multiclass',
        num_class=len(np.unique(y)),
        metric='multi_logloss',
        boosting_type='gbdt', 
        num_leaves=31,
        max_depth=-1,
        learning_rate=0.05,
        n_estimators=500,
        random_state=42,
        verbosity=-1,
        class_weight='balanced'
    ))
]

In [13]:
# Final layer: 1 very strong meta-learner (XGBoost)
final_model = XGB

In [14]:
level2_stack = StackingClassifier(
    estimators=level2_estimators,
    final_estimator=final_model,
    cv=2,
    stack_method='predict_proba',
    n_jobs=-1,
    passthrough=False
)

In [15]:
final_ensemble = StackingClassifier(
    estimators=level1_estimators,
    final_estimator=level2_stack,
    cv=2,
    stack_method='predict_proba',
    n_jobs=-1,
    passthrough=False
)

In [ ]:
final_ensemble.fit(X, y)

In [ ]:
y_pred = final_ensemble.predict(test_)

In [ ]:
y_pred = np.where(y_pred == 0, 'Low',
         np.where(y_pred == 1, 'Medium', 'High'))
submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': y_pred
})
submission.to_csv('1submission.csv', index=False)